In [16]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
import random
from concurrent.futures import ThreadPoolExecutor
import joblib  # Để lưu mô hình chuẩn hóa

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [17]:
selected_columns = ['Total Fwd Packets', 'Total Backward Packets',
       'Fwd Packets Length Total', 'Bwd Packets Length Total',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 
       'CWE Flag Count', 'ECE Flag Count',
       'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 
       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 
       'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
       'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
       'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
       'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
       'Idle Min', 'Label']

len(selected_columns)

65

In [18]:
# df = pd.read_csv('data/csv/cicddos_2019.csv')
# df = pd.read_csv('data/csv/test_benign.csv')
df = pd.read_csv('data/csv/test_syn_flood.csv')

df = df[selected_columns]

In [19]:

# datadir = 'cic_ddos_2019_images'
datadir = 'test_image'
def convert(df_normalized_splited, label, num):
    # Kích thước ảnh ban đầu (5x15) => Mở rộng mỗi điểm thành 3x3 pixel => Ảnh mới (15x45)
    image_size = (8, 8) # kích thước ảnh ban đầu
    upscale_factor = 28 # tỉ lệ tăng kích thước điểm ảnh => size ảnh: (8x28) x (8x28)
    new_image_size = (image_size[0] * upscale_factor, image_size[1] * upscale_factor)

    os.makedirs(f"data/{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu (8x8)
        image_array = np.array(row).reshape(image_size)
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor, upscale_factor))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

        # Chuyển đổi sang ảnh
        image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB
        image = image.convert("RGB")

        ###### thêm nhiễu vào ảnh ######
        rotate_prob = random.random() < 0.5 # xoay
        flip_prob = random.random() < 0.5 # lật
        blur_prob = random.random() <= 0.5  # xác suất làm mờ
        noise_prob = random.random() <= 0.5  # xác suất thêm nhiễu

        # Xoay ảnh ngẫu nhiên
        if rotate_prob:  
            angle = random.uniform(-90, 90)  
            image = image.rotate(angle)

        # Lật ảnh ngẫu nhiên
        if flip_prob:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)  # Lật ngang
            else:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)  # Lật dọc

        # làm mờ ảnh
        if blur_prob:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(10, 20))) # mức độ mờ

        #làm nhiễu ảnh
        if noise_prob:
            noise = np.random.normal(25, 50, (new_image_size[0], new_image_size[1]))  # Thêm nhiễu Gaussian
            noisy_image_array = np.array(image.convert("L")) + noise  # Chuyển sang grayscale trước khi thêm nhiễu
            noisy_image_array = np.clip(noisy_image_array, 0, 255).astype(np.uint8)  # Giữ giá trị trong khoảng 0-255
            image = Image.fromarray(noisy_image_array).convert("RGB")

        # Lưu ảnh
        image.save(f"data/{datadir}/{label}/{str(i)}.png")

        i += 1
        if i > num:
            break

def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/{datadir}/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 500
    convert(df_normalized, label, num=n)

In [20]:
df1 = df.drop(columns=['Label'])
df2 = df['Label']

df1.replace([-np.inf, np.inf], 0, inplace=True)
df1.fillna(df1.mean(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

standardScaler = StandardScaler()
standardScaler.fit(df1) 
joblib.dump(standardScaler, "scaler/standardScaler.pkl")

df1 = standardScaler.fit_transform(df1)

df1 = np.log1p(df1 + 1)
df1 = pd.DataFrame(MinMaxScaler(feature_range=(0, 255)).fit_transform(df1).astype(np.uint8))

grouped =  pd.concat([df1, df2], ignore_index=True)
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

# Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

def process_label(label, df_label):
    df_drop_label = df_label.drop(columns=['Label'])
    data_normalized = df_drop_label
    setup_to_convert(data_normalized, label)

# Sử dụng ThreadPoolExecutor để chạy đa luồng với tối đa 5 luồng
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_label, label, df_label) for label, df_label in dfs.items()]

    # Đợi tất cả các task hoàn thành
    for future in futures:
        future.result()

print("Hoàn thành xử lý đa luồng!")

C:\Users\NewTun\AppData\Local\Temp\ipykernel_18904\4142568662.py:22: RuntimeWarning: invalid value encountered in cast
  image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB


Hoàn thành xử lý đa luồng!
